Démonstration du problème du choix du maximum après un pooling, afin d'assurer sa restitution par un unpooling.

In [1]:
import torch
import torch.nn as nn

In [14]:
# Création d'un tenseur de dimension (5, 5) avec des nombres aléatoires
tensor = torch.rand((5, 5))

# Assurer que le maximum soit en position (1, 2)
tensor[1, 2] = tensor.max() + 1

print("Tenseur original:")
print(tensor)

# Application de la fonction maxpool
maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=0, return_indices=True)
tensor_reshaped = tensor.unsqueeze(0).unsqueeze(0)  # Reshape pour correspondre aux dimensions attendues par MaxPool2d
pooled_tensor, indices  = maxpool(tensor_reshaped)

print("Tenseur après maxpool:")
print(pooled_tensor.squeeze())
print(indices.squeeze())

Tenseur original:
tensor([[0.3551, 0.4701, 0.8251, 0.4125, 0.7306],
        [0.8537, 0.6908, 1.9080, 0.5459, 0.4654],
        [0.5347, 0.0658, 0.8906, 0.4128, 0.6267],
        [0.3513, 0.2492, 0.6619, 0.2765, 0.7115],
        [0.3285, 0.9080, 0.8236, 0.5499, 0.6318]])
Tenseur après maxpool:
tensor([[1.9080, 1.9080],
        [0.9080, 0.8906]])
tensor([[ 7,  7],
        [21, 12]])


In [15]:
pooled_tensor.size()

torch.Size([1, 1, 2, 2])

On ne garde pas la dernière mention (ligne max, colonne max) : ici, la première.

In [16]:
cleaned_tensor = torch.zeros_like(pooled_tensor)
cleaned_tensor[0, 0, 0, 0] = tensor.max().item()
cleaned_tensor

tensor([[[[1.9080, 0.0000],
          [0.0000, 0.0000]]]])

In [17]:
maxunpool = nn.MaxUnpool2d(
    kernel_size=maxpool.kernel_size,
    stride=maxpool.stride,
    padding=maxpool.padding
)

print(maxunpool(cleaned_tensor, indices))

tensor([[[[0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0.]]]])


Constat de l'écrasement du max restitué, par 0. Maintenant, on garde la dernière position du max.

In [19]:
cleaned_tensor = torch.zeros_like(pooled_tensor)
cleaned_tensor[0, 0, 0, 1] = tensor.max().item()
cleaned_tensor


tensor([[[[0.0000, 1.9080],
          [0.0000, 0.0000]]]])

In [20]:
print(maxunpool(cleaned_tensor, indices))

tensor([[[[0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
          [0.0000, 0.0000, 1.9080, 0.0000, 0.0000],
          [0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
          [0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
          [0.0000, 0.0000, 0.0000, 0.0000, 0.0000]]]])


Le max est restitué.